In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
DATA_DIR = Path("data")

COUNTRY_FILES = {
    "US": DATA_DIR / "USvideos.csv",
    "GB": DATA_DIR / "GBvideos.csv",
    "CA": DATA_DIR / "CAvideos.csv",
    "IN": DATA_DIR / "INvideos.csv",
}

available_files = {
    country: path
    for country, path in COUNTRY_FILES.items()
    if path.exists()
}

if not available_files:
    raise FileNotFoundError(
        "No country CSV files found in the local data/ folder."
    )

print("Available countries:", list(available_files))

# Load each country and attach its country code.
country_frames = {}

for country, path in available_files.items():
    df_country = pd.read_csv(path)
    df_country["country"] = country
    country_frames[country] = df_country

# Start with the first available country for the NumPy exercises.
df = next(iter(country_frames.values())).copy()

likes = pd.to_numeric(df["likes"], errors="coerce").fillna(0).to_numpy()
dislikes = pd.to_numeric(df["dislikes"], errors="coerce").fillna(0).to_numpy()
views = pd.to_numeric(df["views"], errors="coerce").fillna(0).to_numpy()
comments = pd.to_numeric(
    df["comment_count"], errors="coerce"
).fillna(0).to_numpy()

for name, arr in [
    ("likes", likes),
    ("dislikes", dislikes),
    ("views", views),
    ("comments", comments),
]:
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

Available countries: ['US', 'GB', 'IN']
likes: shape=(16080,), dtype=float64
dislikes: shape=(16080,), dtype=int64
views: shape=(16080,), dtype=int64
comments: shape=(16080,), dtype=float64


In [3]:

# Safe division: rows with zero total likes + dislikes get 0.
engagement_total = likes + dislikes

like_share = np.divide(
    likes,
    engagement_total,
    out=np.zeros_like(engagement_total, dtype=float),
    where=engagement_total > 0,
)

# Min-max normalization.
def minmax_scale(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x.copy()

    x_min = x.min()
    x_max = x.max()

    if x_max == x_min:
        return np.zeros_like(x, dtype=float)

    return (x - x_min) / (x_max - x_min)


# Normalize each metric so one large-scale metric doesn't dominate.
views_scaled = minmax_scale(views)
likes_scaled = minmax_scale(likes)
comments_scaled = minmax_scale(comments)

# Weighted engagement score.
score = (
    0.40 * views_scaled
    + 0.35 * likes_scaled
    + 0.25 * comments_scaled
)

# Log transform to reduce skew in views.
log_views = np.log1p(views)

# Top 10% based on the 90th percentile.
score_cutoff = np.percentile(score, 90) if score.size else np.nan
top_10_percent = score >= score_cutoff if score.size else np.array([], dtype=bool)

df["like_share"] = like_share
df["log_views"] = log_views
df["engagement_score"] = score
df["top_10_percent"] = top_10_percent

print("90th percentile score:", score_cutoff)
print("Videos at or above cutoff:", int(top_10_percent.sum()))
print(df[["video_id", "views", "engagement_score", "top_10_percent"]].head())


90th percentile score: 0.014499169833980145
Videos at or above cutoff: 1608
      video_id   views  engagement_score  top_10_percent
0  g_N-w8LeAi8   48555          0.000365           False
1  Ugyql2NlPAd  198385          0.001804           False
2  8fE19jkxJ9n  231656          0.001502           False
3  WOY0nn7HCsO    8083          0.000044           False
4  2Lc_Cs5VVr_  373161          0.002432           False


In [4]:

# Combine country data before building the matrix.
all_df = pd.concat(country_frames.values(), ignore_index=True)

# Parse trending_date using the dataset's yy.dd.mm format.
all_df["trending_date"] = pd.to_datetime(
    all_df["trending_date"].astype(str),
    format="%y.%d.%m",
    errors="coerce",
)

# Count records by date and country.
daily_counts = (
    all_df.dropna(subset=["trending_date"])
    .groupby([
        all_df["trending_date"].dt.normalize(),
        "country",
    ])
    .size()
    .rename("video_records")
    .reset_index()
)

daily_counts = daily_counts.rename(
    columns={"trending_date": "date"}
)

# Rows = days; columns = countries.
days_countries = daily_counts.pivot_table(
    index="date",
    columns="country",
    values="video_records",
    aggfunc="sum",
    fill_value=0,
).sort_index()

arr = days_countries.to_numpy(dtype=float)

print("Matrix shape (days, countries):", arr.shape)
print("Countries:", days_countries.columns.tolist())
print("First five rows:\n", arr[:5])

# Broadcasting: divide each row by its daily total.
row_totals = arr.sum(axis=1, keepdims=True)

daily_proportions = np.divide(
    arr,
    row_totals,
    out=np.zeros_like(arr, dtype=float),
    where=row_totals > 0,
)

proportion_df = pd.DataFrame(
    daily_proportions,
    index=days_countries.index,
    columns=days_countries.columns,
)

print("Daily country proportions:\n", proportion_df.head())


Matrix shape (days, countries): (80, 3)
Countries: ['GB', 'IN', 'US']
First five rows:
 [[200. 200. 200.]
 [200. 201. 202.]
 [200. 200. 200.]
 [200. 201. 201.]
 [201. 203. 200.]]
Daily country proportions:
 country           GB        IN        US
date                                    
2017-11-14  0.333333  0.333333  0.333333
2017-11-15  0.331675  0.333333  0.334992
2017-11-16  0.333333  0.333333  0.333333
2017-11-17  0.332226  0.333887  0.333887
2017-11-18  0.332781  0.336093  0.331126


In [5]:

# Keep a copy of the combined raw records for audit purposes.
audit_df = all_df.copy()

# 1. Duplicate records using the required composite key.
duplicate_mask = audit_df.duplicated(
    subset=["video_id", "trending_date", "country"],
    keep=False,
)

print("Rows involved in duplicate keys:", int(duplicate_mask.sum()))
print(
    audit_df.loc[
        duplicate_mask,
        ["video_id", "trending_date", "country"],
    ].head(10)
)

# 2. Missing descriptions.
if "description" in audit_df.columns:
    missing_description = (
        audit_df["description"].isna()
        | audit_df["description"].astype(str).str.strip().eq("")
    )
    print("Missing/blank descriptions:", int(missing_description.sum()))
else:
    print("Description column is absent; missingness cannot be assessed.")

# 3. Invalid dates: check the original strings separately.
invalid_date_counts = {}

for country, path in available_files.items():
    raw_dates = pd.read_csv(path, usecols=["trending_date"])["trending_date"]
    parsed_dates = pd.to_datetime(
        raw_dates.astype(str),
        format="%y.%d.%m",
        errors="coerce",
    )
    invalid_date_counts[country] = int(parsed_dates.isna().sum())

print("Invalid trending dates by country:", invalid_date_counts)

# 4. Likely deleted/private video placeholders.
# These are title-based indicators, not definitive proof of removal.
if "title" in audit_df.columns:
    title_text = audit_df["title"].fillna("").astype(str)

    removed_mask = title_text.str.contains(
        r"\[(?:deleted|private) video\]",
        case=False,
        regex=True,
    )

    print("Likely removed/private video placeholders:", int(removed_mask.sum()))
    print(
        audit_df.loc[removed_mask, ["video_id", "title", "country"]]
        .head(10)
    )
else:
    print("Title column is absent; removal placeholders cannot be checked.")

# 5. Date conversion issues in the combined dataset.
print("Missing parsed trending dates:", int(audit_df["trending_date"].isna().sum()))


Rows involved in duplicate keys: 460
         video_id trending_date country
226   5mZQjUmM993    2017-11-15      US
227   5mZQjUmM993    2017-11-15      US
251   2-WgN1OMDpr    2017-11-15      US
252   2-WgN1OMDpr    2017-11-15      US
728   bMzg4V076SI    2017-11-17      US
729   bMzg4V076SI    2017-11-17      US
1359  L8e0D8nU9wg    2017-11-20      US
1360  L8e0D8nU9wg    2017-11-20      US
1486  XMVR2BV-8jF    2017-11-21      US
1487  XMVR2BV-8jF    2017-11-21      US
Missing/blank descriptions: 1624
Invalid trending dates by country: {'US': 0, 'GB': 0, 'IN': 0}
Likely removed/private video placeholders: 0
Empty DataFrame
Columns: [video_id, title, country]
Index: []
Missing parsed trending dates: 0


In [6]:

# Question 1: Which videos have at least one million views?
high_view_videos = all_df[
    pd.to_numeric(all_df["views"], errors="coerce") >= 1_000_000
]
print("Videos with >= 1M views:", len(high_view_videos))

# Question 2: Which videos have more likes than dislikes?
likes_numeric = pd.to_numeric(all_df["likes"], errors="coerce")
dislikes_numeric = pd.to_numeric(all_df["dislikes"], errors="coerce")

more_likes = all_df[
    (likes_numeric > dislikes_numeric)
    & likes_numeric.notna()
    & dislikes_numeric.notna()
]
print("Videos with more likes than dislikes:", len(more_likes))

# Question 3: Which records have a valid trending date?
valid_dates = all_df[all_df["trending_date"].notna()]
print("Records with valid dates:", len(valid_dates))

# Question 4: Which titles contain the word "music"?
music_videos = all_df[
    all_df["title"].fillna("").str.contains(
        "music", case=False, regex=False
    )
]
print("Titles containing music:", len(music_videos))


Videos with >= 1M views: 9740
Videos with more likes than dislikes: 45635
Records with valid dates: 46230
Titles containing music: 0


In [7]:

category_path = DATA_DIR / "categories.csv"

if not category_path.exists():
    raise FileNotFoundError(
        "categories.csv is missing. Convert US_category_id.json to CSV first."
    )

categories = pd.read_csv(category_path)

# Align join-key types.
all_df["category_id"] = all_df["category_id"].astype("string")
categories["category_id"] = categories["category_id"].astype("string")

# Remove duplicate lookup keys to avoid accidentally multiplying video rows.
categories = categories.drop_duplicates(
    subset=["category_id"],
    keep="first",
)

before_merge = len(all_df)

merged_df = all_df.merge(
    categories[["category_id", "category_name"]],
    on="category_id",
    how="left",
    validate="many_to_one",
)

after_merge = len(merged_df)

print("Rows before merge:", before_merge)
print("Rows after merge:", after_merge)
print("Rows preserved:", before_merge == after_merge)
print("Missing category names:", int(merged_df["category_name"].isna().sum()))


Rows before merge: 46230
Rows after merge: 46230
Rows preserved: True
Missing category names: 0


In [8]:

merged_df["views"] = pd.to_numeric(merged_df["views"], errors="coerce")
merged_df["likes"] = pd.to_numeric(merged_df["likes"], errors="coerce")
merged_df["comment_count"] = pd.to_numeric(
    merged_df["comment_count"], errors="coerce"
)

country_summary = merged_df.groupby("country").agg(
    record_count=("video_id", "size"),
    unique_videos=("video_id", "nunique"),
    average_views=("views", "mean"),
    median_views=("views", "median"),
    total_likes=("likes", "sum"),
    total_comments=("comment_count", "sum"),
).reset_index()

print(country_summary)


  country  record_count  unique_videos  average_views  median_views  \
0      GB         15075           4113   1.580015e+06      316900.0   
1      IN         15075           5878   9.545924e+05      355802.0   
2      US         16080           5068   1.012289e+06      289072.5   

   total_likes  total_comments  
0  981076881.0      60885691.0  
1  500633501.0      48361257.0  
2  635076553.0      50111882.0  


In [11]:

category_summary = merged_df.groupby(
    ["country", "category_name"],
    dropna=False,
).agg(
    record_count=("video_id", "size"),
    average_views=("views", "mean"),
    median_views=("views", "median"),
).reset_index()

category_summary = category_summary.sort_values(
    ["country", "record_count"],
    ascending=[True, False],
)

print(category_summary.head(20))


   country         category_name  record_count  average_views  median_views
7       GB                 Music          5013   3.719037e+06      891864.0
3       GB         Entertainment          2844   8.167331e+05      290092.5
9       GB        People & Blogs          1299   4.369501e+05      220497.0
1       GB                Comedy           941   5.031824e+05      246935.0
4       GB      Film & Animation           714   7.947866e+05      190517.0
12      GB                Sports           678   2.841837e+05      158520.5
5       GB                Gaming           607   4.968345e+05      236381.0
6       GB         Howto & Style           588   2.182217e+05      142459.0
8       GB       News & Politics           536   2.334185e+05      122501.0
2       GB             Education           471   2.568796e+05      123459.0
11      GB  Science & Technology           470   3.326311e+05      181208.5
0       GB      Autos & Vehicles           345   2.222824e+05      123022.0
10      GB  

In [13]:

# Convert both columns to datetime first
merged_df["trending_date"] = pd.to_datetime(
    merged_df["trending_date"],
    errors="coerce"
)

merged_df["publish_time"] = pd.to_datetime(
    merged_df["publish_time"],
    errors="coerce",
    utc=True
)

# Convert publication timestamps to timezone-naive UTC
publish_date = (
    merged_df["publish_time"]
    .dt.tz_convert(None)
    .dt.normalize()
)

trend_date = merged_df["trending_date"].dt.normalize()

# Calculate days between publication and trending
merged_df["days_to_trend"] = (
    trend_date - publish_date
).dt.days

# Display results
print(
    merged_df[
        ["video_id", "publish_time", "trending_date", "days_to_trend"]
    ].head(10)
)

print(
    "Missing days_to_trend:",
    int(merged_df["days_to_trend"].isna().sum())
)

print(
    "Negative days_to_trend:",
    int((merged_df["days_to_trend"] < 0).sum())
)


      video_id              publish_time trending_date  days_to_trend
0  g_N-w8LeAi8 2017-11-14 07:04:28+00:00    2017-11-14              0
1  Ugyql2NlPAd 2017-11-12 22:35:48+00:00    2017-11-14              2
2  8fE19jkxJ9n 2017-11-13 08:40:22+00:00    2017-11-14              1
3  WOY0nn7HCsO 2017-10-27 22:21:47+00:00    2017-11-14             18
4  2Lc_Cs5VVr_ 2017-11-09 13:20:20+00:00    2017-11-14              5
5  UQtb4uZmkX8 2017-11-14 02:12:04+00:00    2017-11-14              0
6  9NxCl722uu2 2017-11-14 09:35:32+00:00    2017-11-14              0
7  3KkxacvDZi1 2017-11-14 11:04:57+00:00    2017-11-14              0
8  RYPa_b_cHgg 2017-11-13 17:03:39+00:00    2017-11-14              1
9  GdoafB2Xgcv 2017-11-13 03:27:13+00:00    2017-11-14              1
Missing days_to_trend: 0
Negative days_to_trend: 0


In [14]:

lifespan = (
    merged_df
    .groupby(["country", "video_id"])["trending_date"]
    .nunique()
    .reset_index(name="trending_days")
)

print(lifespan.head(10))

print("Longest trending appearances:")
print(
    lifespan.sort_values("trending_days", ascending=False).head(10)
)


  country     video_id  trending_days
0      GB  --UQXlrk5E0              4
1      GB  --pb5oKkYrH              4
2      GB  -13iTcoyyzU              1
3      GB  -1VjLKhzB3I              1
4      GB  -1aX3Wk1cJx              4
5      GB  -1rsv0uY-8K              3
6      GB  -3WfJZgVB5h              4
7      GB  -48JdRO2P5_              2
8      GB  -5gYZ79gyl-              1
9      GB  -5iZ5HWgnCs              4
Longest trending appearances:
      country     video_id  trending_days
1611       GB  NqokCmZykzK             25
1630       GB  O6zdCRwWdf1             25
1019       GB  EZaG9_2Yi1u             25
950        GB  DYLzrqG-SrK             25
1846       GB  RdqSDDng84n             25
1432       GB  Kv1WBQp86cP             25
3859       GB  w1lZB-FX3Sn             25
11006      US  BsTMdCTYs7P             25
2343       GB  Z1aFIGyR5zv             25
2687       GB  dIbJkzpClRM             25
